> **Executed on Databricks** (Free Edition, serverless compute, Unity Catalog).  
> Job run `814281740296490` · task `t09_audit_and_validation` · notebook `03_audit_and_validation` · started 2026-10-11 13:11 UTC · result **SUCCESS**  
> Parameters: `catalog=workspace`

Saved with `scripts/export_databricks_run.py`: the cells and outputs below are exactly what the run produced.

# 03 · Audit & validation: Phase 2 evidence
Every section below is computed from the live Unity Catalog tables. They follow the rubric's proof list:

| # | Proof | Section |
|---|---|---|
| 1 | Explicit schemas, no inference | 1. Raw contracts as StructType, and the code that reads files with them |
| 2 | Data model | 2. `DESCRIBE TABLE` for every Bronze and Silver table, and their primary keys |
| 3 | `load_timestamp` and source tracking | 3. Null checks on every table, and Bronze lineage columns |
| 4 | MERGE INTO | 4. The exact MERGE statement the pipeline runs, and what Delta recorded |
| 5 | Re-run creates no duplicates | 5. Duplicate-key check, and MERGE metrics of the re-run |
| 6 | Backfill | 6. Log rows of the backfill runs |
| 7 | Schema change handled | 7. Drift events, quarantine and the evolved column |
| 8 | Log table | 8. `so_ops.pipeline_execution_logs` and the batch registry |

In [1]:
dbutils.widgets.text("catalog", "workspace")

In [2]:
%run ./_bootstrap

repo root: /Workspace/Users/<your-email>/stackoverflow-lakehouse


In [3]:
import inspect

from pyspark.sql import functions as F

from so_lakehouse import bronze as bronze_module
from so_lakehouse.schemas import RAW_CONTRACTS, TABLES
from so_lakehouse.silver import questions_merge_sql

cfg = build_config()
L = cfg.table("ops.pipeline_execution_logs")
BRONZE_SILVER = [k for k, v in TABLES.items() if v[0] in ("bronze", "silver")]

## 1. Explicit schemas (schema-on-read, no inference)
Each raw file type has a `StructType` contract defined in `src/so_lakehouse/schemas.py` **before** any file is read.

In [4]:
for entity, (file_name, contract, pk) in RAW_CONTRACTS.items():
    print(f"=== {entity}  (file: {file_name}, primary key: {', '.join(pk)})")
    spark.createDataFrame([], contract).printSchema()

=== questions  (file: questions.jsonl, primary key: question_id)
root
 |-- question_id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- body: string (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- owner: struct (nullable = true)
 |    |-- account_id: long (nullable = true)
 |    |-- user_id: long (nullable = true)
 |    |-- user_type: string (nullable = true)
 |    |-- reputation: integer (nullable = true)
 |    |-- accept_rate: integer (nullable = true)
 |    |-- display_name: string (nullable = true)
 |    |-- profile_image: string (nullable = true)
 |    |-- link: string (nullable = true)
 |-- is_answered: boolean (nullable = true)
 |-- view_count: integer (nullable = true)
 |-- answer_count: integer (nullable = true)
 |-- score: integer (nullable = true)
 |-- accepted_answer_id: long (nullable = true)
 |-- creation_date: long (nullable = true)
 |-- last_activity_date: long (nullable = true)
 |-- last_edit_da

The code that reads every raw file: plain text lines, then `from_json` with the contract in PERMISSIVE mode.

In [5]:
src = inspect.getsource(bronze_module.load_entity_file)
print(src[src.index("# 1-2."):src.index("counts = ")])

# 1-2. strict parse + contract check ------------------------------------------------
    parse_schema = StructType(list(contract.fields) + [StructField("_corrupt_record", StringType())])
    check = make_contract_check(contract)
    pk_missing = reduce(lambda a, b: a | b, [F.col(f"rec.{k}").isNull() for k in pk])
    parsed = (spark.read.text(path)
              .select(F.col("value"), F.col("_metadata.file_path").alias("_source_file"))
              .where(F.length(F.trim("value")) > 0)
              .withColumn("_record_hash", F.sha2(F.col("value"), 256))
              .withColumn("rec", F.from_json("value", parse_schema,
                                             {"mode": "PERMISSIVE", "columnNameOfCorruptRecord": "_corrupt_record"}))
              .withColumn("chk", check(F.col("value")))
              .withColumn("_reason",
                          F.when(~F.col("chk.is_json"), F.lit("malformed_json"))
                          .when((F.size("chk.type_errors") > 0) | F.col("re

In [6]:
src_dir = os.path.join(REPO_ROOT, "src", "so_lakehouse")
hits = {f: open(os.path.join(src_dir, f)).read().count("inferSchema")
        for f in sorted(os.listdir(src_dir)) if f.endswith(".py")}
print("Occurrences of 'inferSchema' in the pipeline code:", sum(hits.values()), hits)

Occurrences of 'inferSchema' in the pipeline code: 0 {'__init__.py': 0, 'audit.py': 0, 'batches.py': 0, 'bronze.py': 0, 'cli.py': 0, 'config.py': 0, 'ddl.py': 0, 'delta_utils.py': 0, 'demo.py': 0, 'drift.py': 0, 'schemas.py': 0, 'silver.py': 0, 'spark_session.py': 0}


## 2. Data model: `DESCRIBE TABLE` for every Bronze and Silver table

In [7]:
for logical in BRONZE_SILVER:
    print(f"=== {cfg.table(logical)}   primary key: {', '.join(TABLES[logical][3])}")
    display(spark.sql(f"DESCRIBE TABLE {cfg.table(logical)}"))

=== workspace.so_bronze.questions   primary key: question_id, _batch_id


col_name,data_type,comment
question_id,bigint,Question id (natural key)
title,string,"Title, HTML-entity encoded"
body,string,Body HTML (contains free text and possible PII)
tags,array<string>,All tags on the question
owner,"struct<account_id:bigint,user_id:bigint,user_type:string,reputation:int,accept_rate:int,display_name:string,profile_image:string,link:string>",Author snapshot
is_answered,boolean,API flag: has an upvoted or accepted answer
view_count,int,Lifetime views at extraction time
answer_count,int,Number of answers at extraction time
score,int,Upvotes minus downvotes at extraction time
accepted_answer_id,bigint,"Accepted answer id, if any"


=== workspace.so_bronze.answers   primary key: answer_id, _batch_id


col_name,data_type,comment
answer_id,bigint,Answer id (natural key)
question_id,bigint,Parent question id
body,string,Body HTML (contains free text and possible PII)
owner,"struct<account_id:bigint,user_id:bigint,user_type:string,reputation:int,accept_rate:int,display_name:string,profile_image:string,link:string>",Author snapshot
is_accepted,boolean,Whether this is the accepted answer
score,int,Score at extraction time
creation_date,bigint,"Creation time, epoch seconds"
last_activity_date,bigint,"Last activity, epoch seconds"
last_edit_date,bigint,"Last edit, epoch seconds"
locked_date,bigint,"Lock time, epoch seconds"


=== workspace.so_bronze.deleted_questions   primary key: question_id, _batch_id


...

col_name,data_type,comment
question_id,bigint,Question id
snapshot_at,timestamp,"Observation time, UTC"
snapshot_date,date,"Observation date, UTC"
score,int,Score at snapshot
view_count,int,Views at snapshot
answer_count,int,Answers at snapshot
is_answered_api,boolean,API answered flag at snapshot
closed_at,timestamp,Closure time if closed
source_batch_id,string,Batch the snapshot came from
load_timestamp,timestamp,When inserted into Silver


=== workspace.so_silver.post_text_masked   primary key: post_type, post_id


col_name,data_type,comment
post_type,string,question or answer
post_id,bigint,question_id or answer_id
title_masked,string,Decoded title with PII masked (questions only)
body_masked,string,"Body HTML with emails, IPs, secrets and connection strings masked"
masked_token_count,int,How many values were masked
source_extracted_at,timestamp,Extraction time of the current version
record_hash,string,SHA-256 of the masked text
load_timestamp,timestamp,When last inserted or updated in Silver


=== workspace.so_silver.dq_quarantine   primary key: dq_id


col_name,data_type,comment
dq_id,string,"SHA-256 of entity, key, rule and batch"
entity,string,questions or answers
record_key,string,Natural key of the failing record
rule,string,Data-quality rule that failed
source_batch_id,string,Batch of the failing record
payload,string,JSON of the record as it would have entered Silver
load_timestamp,timestamp,When the record was quarantined


Primary keys declared in Unity Catalog (informational constraints created by `00_setup`).

In [8]:
display(spark.sql(f"""
  SELECT kcu.table_schema, kcu.table_name, tc.constraint_name,
         concat_ws(', ', collect_list(kcu.column_name)) AS primary_key_columns
  FROM {cfg.catalog}.information_schema.table_constraints tc
  JOIN {cfg.catalog}.information_schema.key_column_usage kcu
    ON tc.constraint_name = kcu.constraint_name AND tc.table_schema = kcu.table_schema
  WHERE tc.constraint_type = 'PRIMARY KEY' AND tc.table_schema IN ('so_bronze', 'so_silver')
  GROUP BY kcu.table_schema, kcu.table_name, tc.constraint_name
  ORDER BY kcu.table_schema, kcu.table_name"""))

table_schema,table_name,constraint_name,primary_key_columns
so_bronze,answers,pk_bronze_answers,"_batch_id, answer_id"
so_bronze,deleted_questions,pk_bronze_deleted_questions,"_batch_id, question_id"
so_bronze,quarantine_records,pk_bronze_quarantine_records,quarantine_id
so_bronze,question_snapshots,pk_bronze_question_snapshots,"_batch_id, snapshot_at, question_id"
so_bronze,questions,pk_bronze_questions,"question_id, _batch_id"
so_silver,answers,pk_silver_answers,answer_id
so_silver,dq_quarantine,pk_silver_dq_quarantine,dq_id
so_silver,post_text_masked,pk_silver_post_text_masked,"post_type, post_id"
so_silver,question_snapshots,pk_silver_question_snapshots,"question_id, snapshot_at"
so_silver,question_tags,pk_silver_question_tags,"question_id, tag"


## 3. `load_timestamp` on every row, and Bronze source tracking

In [9]:
rows = []
for logical in BRONZE_SILVER:
    df = spark.table(cfg.table(logical))
    r = df.agg(F.count("*").alias("n"), F.sum(F.col("load_timestamp").isNull().cast("int")).alias("nulls"),
               F.min("load_timestamp").alias("first"), F.max("load_timestamp").alias("last")).collect()[0]
    rows.append((cfg.table(logical), r["n"], r["nulls"] or 0, str(r["first"]), str(r["last"])))
display(spark.createDataFrame(rows, "table string, rows long, null_load_timestamps long, earliest string, latest string"))

table,rows,null_load_timestamps,earliest,latest
workspace.so_bronze.questions,7679,0,2026-10-11 13:05:44.003817,2026-10-11 13:09:05.223407
workspace.so_bronze.answers,9701,0,2026-10-11 13:06:00.544682,2026-10-11 13:08:30.054271
workspace.so_bronze.deleted_questions,1,0,2026-10-11 13:07:54.907566,2026-10-11 13:07:54.907566
workspace.so_bronze.question_snapshots,7385,0,2026-10-11 13:07:17.943638,2026-10-11 13:08:04.140096
workspace.so_bronze.quarantine_records,3,0,2026-10-11 13:09:07.575123,2026-10-11 13:09:07.575123
workspace.so_silver.questions,7620,0,2026-10-11 12:59:45.393637,2026-10-11 13:03:41.641195
workspace.so_silver.question_tags,24185,0,2026-10-11 13:00:12.000190,2026-10-11 13:04:02.320025
workspace.so_silver.answers,9611,0,2026-10-11 13:00:22.710431,2026-10-11 13:01:09.528957
workspace.so_silver.users,8332,0,2026-10-11 13:00:32.101059,2026-10-11 13:00:32.101059
workspace.so_silver.question_snapshots,7385,0,2026-10-11 13:00:40.752220,2026-10-11 13:00:40.752220


In [10]:
display(spark.table(cfg.table("bronze.questions"))
        .groupBy("_batch_id", "_load_type", "_batch_date", "_extracted_at", "_source_file")
        .agg(F.count("*").alias("rows"), F.max("load_timestamp").alias("load_timestamp"))
        .orderBy("_extracted_at"))

_batch_id,_load_type,_batch_date,_extracted_at,_source_file,rows,load_timestamp
full_2025-04-01_2025-07-01,full,2026-09-25,2026-09-25T15:58:12.000Z,dbfs:/Volumes/workspace/so_raw/landing/full/full_2025-04-01_2025-07-01/questions.jsonl,2738,2026-10-11T13:05:44.003Z
incr_20260925T155859Z,incremental,2026-09-25,2026-09-25T15:58:59.000Z,dbfs:/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z/questions.jsonl,35,2026-10-11T13:06:52.139Z
full_2025-01-01_2025-04-01,full,2026-10-07,2026-10-07T15:59:01.000Z,dbfs:/Volumes/workspace/so_raw/landing/full/full_2025-01-01_2025-04-01/questions.jsonl,4548,2026-10-11T13:06:20.951Z
incr_20261007T155914Z,incremental,2026-10-07,2026-10-07T16:00:24.000Z,dbfs:/Volumes/workspace/so_raw/landing/incremental/incr_20261007T155914Z/questions.jsonl,305,2026-10-11T13:07:36.186Z
incr_20261007T160033Z,incremental,2026-10-07,2026-10-07T16:00:43.000Z,dbfs:/Volumes/workspace/so_raw/landing/incremental/incr_20261007T160033Z/questions.jsonl,51,2026-10-11T13:08:21.074Z
incr_20261008T000000Z,incremental,2026-10-08,2026-10-08T00:00:00.000Z,dbfs:/Volumes/workspace/so_raw/landing/incremental/incr_20261008T000000Z/questions.jsonl,2,2026-10-11T13:09:05.223Z


## 4. The MERGE INTO statement
This is the exact statement `02_bronze_to_silver` runs for `silver.questions` (built by `questions_merge_sql`).
It updates a row only when the source is **newer and different**, and inserts keys it has never seen.

In [11]:
print(questions_merge_sql(cfg))

MERGE INTO workspace.so_silver.questions AS t
USING src_questions AS s
ON t.question_id = s.question_id
WHEN MATCHED AND (s.source_extracted_at > t.source_extracted_at AND s.record_hash <> t.record_hash) THEN UPDATE SET
    t.title = s.title,
    t.created_at = s.created_at,
    t.last_activity_at = s.last_activity_at,
    t.last_edit_at = s.last_edit_at,
    t.closed_at = s.closed_at,
    t.closed_reason = s.closed_reason,
    t.closed_reason_code = s.closed_reason_code,
    t.is_closed = s.is_closed,
    t.accepted_answer_id = s.accepted_answer_id,
    t.has_accepted_answer = s.has_accepted_answer,
    t.bounty_amount = s.bounty_amount,
    t.tags = s.tags,
    t.tracked_tags = s.tracked_tags,
    t.owner_user_key = s.owner_user_key,
    t.owner_user_type = s.owner_user_type,
    t.owner_reputation = s.owner_reputation,
    t.body_length = s.body_length,
    t.code_block_count = s.code_block_count,
    t.inline_code_count = s.inline_code_count,
    t.link_count = s.link_count,
    t.

What Delta recorded for each MERGE on `silver.questions`: the predicates and the row counts.

In [12]:
display(spark.sql(f"DESCRIBE HISTORY {cfg.table('silver.questions')}")
        .where("operation = 'MERGE'")
        .select("version", "timestamp",
                F.col("operationParameters")["predicate"].alias("on"),
                F.col("operationParameters")["matchedPredicates"].alias("when_matched"),
                F.col("operationMetrics")["numSourceRows"].alias("source_rows"),
                F.col("operationMetrics")["numTargetRowsInserted"].alias("inserted"),
                F.col("operationMetrics")["numTargetRowsUpdated"].alias("updated"))
        .orderBy("version"))

version,timestamp,on,when_matched,source_rows,inserted,updated
1,2026-10-11T12:59:59.000Z,"[""(question_id#235114L = question_id#234819L)""]","[{""predicate"":""((source_extracted_at#234854 > source_extracted_at#235149) AND NOT (record_hash#234855 = record_hash#235150))"",""actionType"":""update""}]",7618,7618,0
2,2026-10-11T13:00:53.000Z,"[""(question_id#246751L = question_id#246482L)""]","[{""predicate"":""(snapshot_at#246483 > metrics_as_of#246784)"",""actionType"":""update""}]",7285,0,7282
4,2026-10-11T13:01:04.000Z,"[""(question_id#250449L = question_id#250181L)""]","[{""predicate"":""(NOT is_deleted#250480 AND (source_extracted_at#250484 < deleted_detected_at#250169))"",""actionType"":""update""}]",1,0,1
5,2026-10-11T13:03:53.000Z,"[""(question_id#270375L = question_id#270080L)""]","[{""predicate"":""((source_extracted_at#270115 > source_extracted_at#270410) AND NOT (record_hash#270116 = record_hash#270411))"",""actionType"":""update""}]",2,2,0
6,2026-10-11T13:04:36.000Z,"[""(question_id#281931L = question_id#281662L)""]","[{""predicate"":""(snapshot_at#281663 > metrics_as_of#281964)"",""actionType"":""update""}]",0,0,0
7,2026-10-11T13:04:42.000Z,"[""(question_id#283491L = question_id#283223L)""]","[{""predicate"":""(NOT is_deleted#283522 AND (source_extracted_at#283526 < deleted_detected_at#283211))"",""actionType"":""update""}]",0,0,0
8,2026-10-11T13:10:07.000Z,"[""(question_id#344192L = question_id#343897L)""]","[{""predicate"":""((source_extracted_at#343932 > source_extracted_at#344227) AND NOT (record_hash#343933 = record_hash#344228))"",""actionType"":""update""}]",7620,0,0
9,2026-10-11T13:10:58.000Z,"[""(question_id#356709L = question_id#355984L)""]","[{""predicate"":""(snapshot_at#355985 > metrics_as_of#356742)"",""actionType"":""update""}]",7285,0,0
10,2026-10-11T13:11:05.000Z,"[""(question_id#358296L = question_id#358028L)""]","[{""predicate"":""(NOT is_deleted#358327 AND (source_extracted_at#358331 < deleted_detected_at#358016))"",""actionType"":""update""}]",1,0,0


## 5. Re-running the same input creates no duplicates
Every Silver primary key must be unique, and the backfill that re-merged every batch must have changed 0 rows.

In [13]:
checks = {"silver.questions": ["question_id"], "silver.answers": ["answer_id"],
          "silver.question_tags": ["question_id", "tag"], "silver.users": ["user_key"],
          "silver.question_snapshots": ["question_id", "snapshot_at"],
          "silver.post_text_masked": ["post_type", "post_id"]}
rows = []
for t, keys in checks.items():
    df = spark.table(cfg.table(t))
    total, distinct = df.count(), df.select(*keys).distinct().count()
    rows.append((t, total, distinct, total - distinct))
display(spark.createDataFrame(rows, "table string, rows long, distinct_keys long, duplicates long"))

table,rows,distinct_keys,duplicates
silver.questions,7620,7620,0
silver.answers,9611,9611,0
silver.question_tags,24185,24185,0
silver.users,8332,8332,0
silver.question_snapshots,7385,7385,0
silver.post_text_masked,17231,17231,0


In [14]:
display(spark.sql(f"""
  SELECT run_id, run_mode, entity, rows_read, rows_inserted, rows_updated, rows_deleted, status
  FROM {L} WHERE layer = 'Bronze-to-Silver' AND status <> 'SKIPPED'
  ORDER BY start_time"""))

run_id,run_mode,entity,rows_read,rows_inserted,rows_updated,rows_deleted,status
bronze_to_silver-20261011T125933Z-d54ec7,incremental,questions,7677,7618,0,0,SUCCESS
bronze_to_silver-20261011T125933Z-d54ec7,incremental,question_tags,24177,24177,0,0,SUCCESS
bronze_to_silver-20261011T125933Z-d54ec7,incremental,answers,9701,9611,0,0,SUCCESS
bronze_to_silver-20261011T125933Z-d54ec7,incremental,users,17280,8332,0,0,SUCCESS
bronze_to_silver-20261011T125933Z-d54ec7,incremental,question_snapshots,7385,7385,0,0,SUCCESS
bronze_to_silver-20261011T125933Z-d54ec7,incremental,question_metrics (from snapshots),7285,0,7282,0,SUCCESS
bronze_to_silver-20261011T125933Z-d54ec7,incremental,deleted_questions -> questions,1,0,1,0,SUCCESS
bronze_to_silver-20261011T125933Z-d54ec7,incremental,deleted_questions -> answers,1,0,2,0,SUCCESS
bronze_to_silver-20261011T125933Z-d54ec7,incremental,post_text_masked,17229,17229,0,0,SUCCESS
bronze_to_silver-20261011T130329Z-5bb8e1,incremental,questions,2,2,0,0,SUCCESS


## 6. Backfill runs (mode = backfill, chosen by date range or batch id)

In [15]:
display(spark.sql(f"""
  SELECT run_id, layer, entity, batch_id, source_parameter, status, rows_read, rows_inserted, rows_updated,
         rows_deleted AS rows_replaced, start_time
  FROM {L} WHERE run_mode = 'backfill' ORDER BY start_time"""))

run_id,layer,entity,batch_id,source_parameter,status,rows_read,rows_inserted,rows_updated,rows_replaced,start_time
raw_to_bronze-20261011T130239Z-8ec00d,Raw-to-Bronze,questions,incr_20261008T000000Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261008T000000Z/questions.jsonl,SUCCESS,5,2,0,0,2026-10-11T13:02:40.851Z
raw_to_bronze-20261011T130525Z-1ec894,Raw-to-Bronze,questions,full_2025-04-01_2025-07-01,/Volumes/workspace/so_raw/landing/full/full_2025-04-01_2025-07-01/questions.jsonl,SUCCESS,2738,2738,0,2738,2026-10-11T13:05:27.541Z
raw_to_bronze-20261011T130525Z-1ec894,Raw-to-Bronze,answers,full_2025-04-01_2025-07-01,/Volumes/workspace/so_raw/landing/full/full_2025-04-01_2025-07-01/answers.jsonl,SUCCESS,3384,3384,0,3384,2026-10-11T13:05:53.304Z
raw_to_bronze-20261011T130525Z-1ec894,Raw-to-Bronze,questions,full_2025-01-01_2025-04-01,/Volumes/workspace/so_raw/landing/full/full_2025-01-01_2025-04-01/questions.jsonl,SUCCESS,4548,4548,0,4548,2026-10-11T13:06:15.172Z
raw_to_bronze-20261011T130525Z-1ec894,Raw-to-Bronze,answers,full_2025-01-01_2025-04-01,/Volumes/workspace/so_raw/landing/full/full_2025-01-01_2025-04-01/answers.jsonl,SUCCESS,5528,5528,0,5528,2026-10-11T13:06:27.261Z
raw_to_bronze-20261011T130525Z-1ec894,Raw-to-Bronze,questions,incr_20260925T155859Z,/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z/questions.jsonl,SUCCESS,35,35,0,35,2026-10-11T13:06:46.723Z
raw_to_bronze-20261011T130525Z-1ec894,Raw-to-Bronze,answers,incr_20260925T155859Z,/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z/answers.jsonl,SUCCESS,59,59,0,59,2026-10-11T13:06:57.785Z
raw_to_bronze-20261011T130525Z-1ec894,Raw-to-Bronze,deleted_questions,incr_20260925T155859Z,/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z/deleted_questions.jsonl,SUCCESS,0,0,0,0,2026-10-11T13:07:07.561Z
raw_to_bronze-20261011T130525Z-1ec894,Raw-to-Bronze,question_snapshots,incr_20260925T155859Z,/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z/question_snapshots.jsonl,SUCCESS,100,100,0,100,2026-10-11T13:07:14.210Z
raw_to_bronze-20261011T130525Z-1ec894,Raw-to-Bronze,questions,incr_20261007T155914Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261007T155914Z/questions.jsonl,SUCCESS,305,305,0,305,2026-10-11T13:07:31.724Z


## 7. Schema change handled: drift events, quarantine and the evolved column

In [16]:
display(spark.table(cfg.table("ops.schema_drift_events"))
        .select("_batch_id", "entity", "drift_type", "column_path", "record_count", "action_taken", "example_value")
        .orderBy("drift_type"))

_batch_id,entity,drift_type,column_path,record_count,action_taken,example_value
incr_20261008T000000Z,questions,malformed_json,,1,quarantined,
incr_20261008T000000Z,questions,missing_primary_key,question_id,1,quarantined,
incr_20261008T000000Z,questions,new_column,owner.badge_counts,1,rescued,"{""gold"": 1, ""silver"": 4}"
incr_20261008T000000Z,questions,new_column,ai_assisted,1,evolved,true
incr_20261008T000000Z,questions,type_mismatch,score,1,quarantined,"expected int, got str: ""five"""


In [17]:
display(spark.table(cfg.table("bronze.quarantine_records"))
        .select("entity", "_batch_id", "reason", "error_detail", "raw_record", "load_timestamp"))

entity    | _batch_id             | reason              | error_detail                             | raw_record                               | load_timestamp          
----------+-----------------------+---------------------+------------------------------------------+------------------------------------------+-------------------------
questions | incr_20261008T000000Z | malformed_json      | line is not a JSON object                | {"question_id": 990000003, "title":      | 2026-10-11T13:09:07.575Z
questions | incr_20261008T000000Z | type_mismatch       | [{"path":"score","expected":"int","actua | {"tags": ["python", "pandas"], "owner":  | 2026-10-11T13:09:07.575Z
questions | incr_20261008T000000Z | missing_primary_key | primary key (question_id) is null        | {"tags": ["sql", "spring-boot", "hsqldb" | 2026-10-11T13:09:07.575Z

In [18]:
bq = spark.table(cfg.table("bronze.questions"))
evolved = [c for c in bq.columns if c not in {f.name for f in TABLES["bronze.questions"][2].fields}]
print("Columns added to bronze.questions by schema evolution (mergeSchema):", evolved)
display(bq.where("_batch_id = 'incr_20261008T000000Z'").select("question_id", "score", *evolved, "_rescued_data"))

Columns added to bronze.questions by schema evolution (mergeSchema): ['ai_assisted']


question_id,score,ai_assisted,_rescued_data
990000001,4,true,"{""owner.badge_counts"":""{\""gold\"": 1, \""silver\"": 4}""}"
990000004,1,,


## 8. The log table: `so_ops.pipeline_execution_logs`
One row per file (Raw-to-Bronze) or target table (Bronze-to-Silver), for full and incremental loads.

In [19]:
display(spark.sql(f"""
  SELECT run_id, layer, run_mode, min(start_time) AS started, max(end_time) AS finished, count(*) AS log_rows,
         sum(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END) AS success,
         sum(CASE WHEN status = 'SKIPPED' THEN 1 ELSE 0 END) AS skipped,
         sum(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) AS failed,
         sum(rows_inserted) AS inserted, sum(rows_updated) AS updated, sum(rows_quarantined) AS quarantined
  FROM {L} GROUP BY run_id, layer, run_mode ORDER BY started"""))

run_id,layer,run_mode,started,finished,log_rows,success,skipped,failed,inserted,updated,quarantined
raw_to_bronze-20261011T125609Z-3c944e,Raw-to-Bronze,incremental,2026-10-11T12:56:11.847Z,2026-10-11T12:59:10.288Z,16,16,0,0,24764,0,0
bronze_to_silver-20261011T125933Z-d54ec7,Bronze-to-Silver,incremental,2026-10-11T12:59:35.140Z,2026-10-11T13:01:35.550Z,9,9,0,0,74352,7285,0
raw_to_bronze-20261011T130158Z-c1c1a6,Raw-to-Bronze,incremental,2026-10-11T13:02:01.012Z,2026-10-11T13:02:08.293Z,5,0,5,0,0,0,0
bronze_to_silver-20261011T130221Z-e73c98,Bronze-to-Silver,incremental,2026-10-11T13:02:23.190Z,2026-10-11T13:02:23.190Z,1,0,1,0,0,0,0
raw_to_bronze-20261011T130239Z-8ec00d,Raw-to-Bronze,backfill,2026-10-11T13:02:40.851Z,2026-10-11T13:03:05.928Z,1,1,0,0,2,0,3
bronze_to_silver-20261011T130329Z-5bb8e1,Bronze-to-Silver,incremental,2026-10-11T13:03:31.235Z,2026-10-11T13:05:03.047Z,9,9,0,0,12,0,0
raw_to_bronze-20261011T130525Z-1ec894,Raw-to-Bronze,backfill,2026-10-11T13:05:27.541Z,2026-10-11T13:09:14.863Z,17,17,0,0,24766,0,3
bronze_to_silver-20261011T130937Z-7f0a33,Bronze-to-Silver,backfill,2026-10-11T13:09:39.959Z,2026-10-11T13:11:37.468Z,9,9,0,0,0,0,0


In [20]:
display(spark.sql(f"""
  SELECT layer, run_mode, load_type, entity, batch_id, source_parameter, start_time, end_time,
         round(duration_seconds, 1) AS secs, status, rows_read, rows_inserted, rows_updated, rows_deleted,
         rows_quarantined, message
  FROM {L} ORDER BY start_time"""))

layer,run_mode,load_type,entity,batch_id,source_parameter,start_time,end_time,secs,status,rows_read,rows_inserted,rows_updated,rows_deleted,rows_quarantined,message
Raw-to-Bronze,incremental,full,questions,full_2025-04-01_2025-07-01,/Volumes/workspace/so_raw/landing/full/full_2025-04-01_2025-07-01/questions.jsonl,2026-10-11T12:56:11.847Z,2026-10-11T12:56:31.755Z,19.9,SUCCESS,2738,2738,0,0,0,
Raw-to-Bronze,incremental,full,answers,full_2025-04-01_2025-07-01,/Volumes/workspace/so_raw/landing/full/full_2025-04-01_2025-07-01/answers.jsonl,2026-10-11T12:56:34.395Z,2026-10-11T12:56:44.276Z,9.9,SUCCESS,3384,3384,0,0,0,
Raw-to-Bronze,incremental,full,questions,full_2025-01-01_2025-04-01,/Volumes/workspace/so_raw/landing/full/full_2025-01-01_2025-04-01/questions.jsonl,2026-10-11T12:56:52.110Z,2026-10-11T12:57:01.749Z,9.6,SUCCESS,4548,4548,0,0,0,
Raw-to-Bronze,incremental,full,answers,full_2025-01-01_2025-04-01,/Volumes/workspace/so_raw/landing/full/full_2025-01-01_2025-04-01/answers.jsonl,2026-10-11T12:57:03.303Z,2026-10-11T12:57:13.061Z,9.8,SUCCESS,5528,5528,0,0,0,
Raw-to-Bronze,incremental,incremental,questions,incr_20260925T155859Z,/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z/questions.jsonl,2026-10-11T12:57:19.730Z,2026-10-11T12:57:26.894Z,7.2,SUCCESS,35,35,0,0,0,
Raw-to-Bronze,incremental,incremental,answers,incr_20260925T155859Z,/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z/answers.jsonl,2026-10-11T12:57:29.131Z,2026-10-11T12:57:35.770Z,6.6,SUCCESS,59,59,0,0,0,
Raw-to-Bronze,incremental,incremental,deleted_questions,incr_20260925T155859Z,/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z/deleted_questions.jsonl,2026-10-11T12:57:37.281Z,2026-10-11T12:57:42.578Z,5.3,SUCCESS,0,0,0,0,0,
Raw-to-Bronze,incremental,incremental,question_snapshots,incr_20260925T155859Z,/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z/question_snapshots.jsonl,2026-10-11T12:57:44.348Z,2026-10-11T12:57:52.213Z,7.9,SUCCESS,100,100,0,0,0,
Raw-to-Bronze,incremental,incremental,questions,incr_20261007T155914Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261007T155914Z/questions.jsonl,2026-10-11T12:57:58.084Z,2026-10-11T12:58:05.752Z,7.7,SUCCESS,305,305,0,0,0,
Raw-to-Bronze,incremental,incremental,answers,incr_20261007T155914Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261007T155914Z/answers.jsonl,2026-10-11T12:58:07.213Z,2026-10-11T12:58:15.044Z,7.8,SUCCESS,646,646,0,0,0,


In [21]:
display(spark.table(cfg.table("ops.batch_registry")).drop("manifest_json").orderBy("extracted_at"))

batch_id,load_type,batch_date,extracted_at,landing_path,bronze_status,bronze_loaded_at,silver_status,silver_processed_at,load_timestamp
full_2025-04-01_2025-07-01,full,2026-09-25,2026-09-25T15:58:12.000Z,/Volumes/workspace/so_raw/landing/full/full_2025-04-01_2025-07-01,SUCCESS,2026-10-11T13:06:06.443Z,SUCCESS,2026-10-11T13:11:38.855Z,2026-10-11T13:11:38.855Z
incr_20260925T155859Z,incremental,2026-09-25,2026-09-25T15:58:59.000Z,/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z,SUCCESS,2026-10-11T13:07:22.429Z,SUCCESS,2026-10-11T13:11:38.855Z,2026-10-11T13:11:38.855Z
full_2025-01-01_2025-04-01,full,2026-10-07,2026-10-07T15:59:01.000Z,/Volumes/workspace/so_raw/landing/full/full_2025-01-01_2025-04-01,SUCCESS,2026-10-11T13:06:39.534Z,SUCCESS,2026-10-11T13:11:38.855Z,2026-10-11T13:11:38.855Z
incr_20261007T155914Z,incremental,2026-10-07,2026-10-07T16:00:24.000Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261007T155914Z,SUCCESS,2026-10-11T13:08:08.978Z,SUCCESS,2026-10-11T13:11:38.855Z,2026-10-11T13:11:38.855Z
incr_20261007T160033Z,incremental,2026-10-07,2026-10-07T16:00:43.000Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261007T160033Z,SUCCESS,2026-10-11T13:08:50.684Z,SUCCESS,2026-10-11T13:11:38.855Z,2026-10-11T13:11:38.855Z
incr_20261008T000000Z,incremental,2026-10-08,2026-10-08T00:00:00.000Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261008T000000Z,SUCCESS,2026-10-11T13:09:16.454Z,SUCCESS,2026-10-11T13:11:38.855Z,2026-10-11T13:11:38.855Z
